In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import random
import numpy as np
from tqdm import tqdm
from collections import deque


torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
# Dataset and DataLoader
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

Files already downloaded and verified
Files already downloaded and verified


In [3]:
train_dataset.data = train_dataset.data[:4096]
train_dataset.targets = train_dataset.targets[:4096]

test_dataset.data = test_dataset.data[:1024]
test_dataset.targets = test_dataset.targets[:1024]

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # output: 64 x 16 x 16

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # output: 128 x 8 x 8

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # output: 256 x 4 x 4

            nn.Flatten(), 
            nn.Linear(256*4*4, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 10))
        
    def forward(self, xb):
        return self.network(xb)

In [5]:
def train_cnn(cnn_model, loader, criterion, optimizer):
    cnn_model.train()
    total_loss, total_correct = 0, 0
    for epoch in range(100):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = cnn_model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            total_correct += (outputs.argmax(dim=1) == labels).sum().item()
    return total_loss / len(loader), total_correct / len(loader.dataset)

def validate_cnn(cnn_model, loader):
    cnn_model.eval()
    total_correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = cnn_model(images)
            total_correct += (outputs.argmax(dim=1) == labels).sum().item()
    return total_correct / len(loader.dataset)

In [6]:
train_cnn_model = CNN().to(device)
train_cnn_criterion = nn.CrossEntropyLoss()
train_cnn_optimizer = optim.Adam(train_cnn_model.parameters(), lr=0.001)
loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
train_cnn(train_cnn_model, loader, train_cnn_criterion, train_cnn_optimizer)
accuracy = validate_cnn(train_cnn_model, test_loader)
accuracy

0.5361328125